In [18]:
def dataset(text, train: float = 0.8, test: float = 0.1, validation: float = 0.1):
    train_split = int(len(text) * train)
    test_split= int(len(text) * (train + test))
    
    train_data = text[:train_split]
    test_data = text[train_split:test_split]
    val_data = text[test_split:]

    return train_data, test_data, val_data

In [2]:
def batcher(encoded_ids, block_size: int = 256, batch_size: int = 32):
    X = []
    targets = []

    for i in range(batch_size):
        ix = torch.randint(0, len(encoded_ids) - (block_size + 1), (1, ))

        X.append(encoded_ids[ix:ix+block_size])
        targets.append(encoded_ids[ix+1:ix+block_size+1])

    X = torch.tensor(X)
    targets = torch.tensor(targets)

    return X, targets

In [3]:
def get_stats(ids):

    stats = {}
    
    for pair in zip(ids, ids[1:]):
        if pair in stats:
            stats[pair] += 1
        else:
            stats[pair] = 1

    return stats

In [4]:
def merge(ids, pair, idx):

    new_ids = []

    ix = 0
    while ix < len(ids):
        
        if pair[0] == ids[ix] and ix < len(ids) - 1 and pair[1] == ids[ix+1]:
            new_ids.append(idx)
            ix += 2

        else:
            new_ids.append(ids[ix])
            ix += 1
    return new_ids

In [5]:
def train_bpe(text, num_merge):
    ids = text.encode('utf-8')
    
    merges = {}
    idx = 256
    for i in range(num_merge):
        stats = get_stats(ids)
        if not stats:
            break        
        
        target_pair = max(stats, key=stats.get)
        ids = merge(ids, target_pair, idx)
        merges[target_pair] = idx
        idx += 1

    return merges

In [6]:
def encoder(text, merges):
    encoded_ids = text.encode('utf-8')

    for pair, token_id in merges.items():
        encoded_ids = merge(encoded_ids, pair, token_id)

    return encoded_ids

In [7]:
def decoder(encoded_ids, vocab):
    text = []
    for idx in encoded_ids:
        ch = vocab[idx]
        text.append(ch)

    text = b''.join(text)
    text = text.decode('utf-8')
    
    return text

In [8]:
def create_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for pair, token_id in merges.items():
        vocab[token_id] = vocab[pair[0]] + vocab[pair[1]]

    return vocab

In [9]:
with open("../data/interim/Bianet_tr_minimal-cleaned.txt", mode='r', encoding="utf-8") as file:
    data = file.read()

In [74]:
class BPETokenizer():
    def fit(self, text, num_merge):
        self.text = text
        ids = text.encode('utf-8')
        merges = {}
        idx = 256
        
        for i in range(num_merge):
            stats = get_stats(ids)
            if not stats:
                break        
            
            target_pair = max(stats, key=stats.get)
            ids = merge(ids, target_pair, idx)
            merges[target_pair] = idx
            idx += 1

        vocab = {i: bytes([i]) for i in range(256)}
        for pair, token_id in merges.items():
            vocab[token_id] = vocab[pair[0]] + vocab[pair[1]]
            
        self.merges = merges
        self.vocab = vocab
        self.vocab_size = len(vocab)

    def encode(self, text):
        encoded_ids = text.encode('utf-8')
    
        for pair, token_id in self.merges.items():
            encoded_ids = merge(encoded_ids, pair, token_id)

        return encoded_ids

    def decode(self, encoded_ids):
        text = []
        
        for idx in encoded_ids:
            ch = self.vocab[idx]
            text.append(ch)
    
        text = b''.join(text)
        text = text.decode('utf-8')
        
        return text

In [143]:
class CharachterTokenizer():
    def fit(self, text):
        chars = sorted(list(set(text)))
        self.itos = {idx: ch for idx, ch in enumerate(chars)}
        self.stoi = {ch: idx for idx, ch in enumerate(chars)}
        
        self.unk_ids = len(chars)
        self.itos[self.unk_ids] = '<UNK>'
        self.stoi['<UNK>'] = self.unk_ids
        self.vocab_size = len(self.itos)

    def encode(self, text):
        ids = []
        for ch in text:
            if ch in self.stoi.keys():
                idx = self.stoi[ch]
                ids.append(idx)
            
            else:
                ids.append(self.unk_ids)

        return ids

    def decode(self, ids):
        text = []
        for idx in ids:
            if idx in self.itos.keys():
                ch = self.itos[idx]
                text.append(ch)
            else:
                idx = self.unk_ids
                ch = self.itos[idx]
                text.append(ch)

        text = ''.join(text)

        return text

In [144]:
sample = data[10000:30000]
train_data, test_data, val_data = dataset(sample) 

In [145]:
len(train_data), len(test_data), len(val_data)

(16000, 2000, 2000)

In [146]:
len(train_data) + len(test_data) + len(val_data) == len(sample)

True

In [147]:
tokenizer_1 = BPETokenizer()
tokenizer_1.fit(train_data, 100)

In [148]:
tokenizer_1.merges == 100

False

In [149]:
encoded_train = tokenizer_1.encode(train_data)

In [150]:
len(tokenizer_1.merges ) + 256 == len(tokenizer_1.vocab)

True

In [151]:
ch_tokenizer = CharachterTokenizer()
ch_tokenizer.fit(train_data)

In [152]:
ch_tokenizer.itos

{0: '\n',
 1: ' ',
 2: '"',
 3: '%',
 4: "'",
 5: '(',
 6: ')',
 7: '*',
 8: ',',
 9: '-',
 10: '.',
 11: '/',
 12: '0',
 13: '1',
 14: '2',
 15: '3',
 16: '4',
 17: '5',
 18: '6',
 19: '7',
 20: '8',
 21: '9',
 22: ':',
 23: ';',
 24: '?',
 25: 'A',
 26: 'B',
 27: 'C',
 28: 'D',
 29: 'E',
 30: 'F',
 31: 'G',
 32: 'H',
 33: 'K',
 34: 'M',
 35: 'N',
 36: 'O',
 37: 'P',
 38: 'R',
 39: 'S',
 40: 'T',
 41: 'U',
 42: 'V',
 43: 'W',
 44: 'Y',
 45: 'Z',
 46: 'a',
 47: 'b',
 48: 'c',
 49: 'd',
 50: 'e',
 51: 'f',
 52: 'g',
 53: 'h',
 54: 'i',
 55: 'j',
 56: 'k',
 57: 'l',
 58: 'm',
 59: 'n',
 60: 'o',
 61: 'p',
 62: 'r',
 63: 's',
 64: 't',
 65: 'u',
 66: 'v',
 67: 'w',
 68: 'y',
 69: 'z',
 70: 'Ç',
 71: 'Ö',
 72: 'Ü',
 73: 'ç',
 74: 'ö',
 75: 'ü',
 76: 'ğ',
 77: 'İ',
 78: 'ı',
 79: 'Ş',
 80: 'ş',
 81: '‘',
 82: '’',
 83: '“',
 84: '”',
 85: '…',
 86: '<UNK>'}

In [153]:
ch_tokenizer.stoi

{'\n': 0,
 ' ': 1,
 '"': 2,
 '%': 3,
 "'": 4,
 '(': 5,
 ')': 6,
 '*': 7,
 ',': 8,
 '-': 9,
 '.': 10,
 '/': 11,
 '0': 12,
 '1': 13,
 '2': 14,
 '3': 15,
 '4': 16,
 '5': 17,
 '6': 18,
 '7': 19,
 '8': 20,
 '9': 21,
 ':': 22,
 ';': 23,
 '?': 24,
 'A': 25,
 'B': 26,
 'C': 27,
 'D': 28,
 'E': 29,
 'F': 30,
 'G': 31,
 'H': 32,
 'K': 33,
 'M': 34,
 'N': 35,
 'O': 36,
 'P': 37,
 'R': 38,
 'S': 39,
 'T': 40,
 'U': 41,
 'V': 42,
 'W': 43,
 'Y': 44,
 'Z': 45,
 'a': 46,
 'b': 47,
 'c': 48,
 'd': 49,
 'e': 50,
 'f': 51,
 'g': 52,
 'h': 53,
 'i': 54,
 'j': 55,
 'k': 56,
 'l': 57,
 'm': 58,
 'n': 59,
 'o': 60,
 'p': 61,
 'r': 62,
 's': 63,
 't': 64,
 'u': 65,
 'v': 66,
 'w': 67,
 'y': 68,
 'z': 69,
 'Ç': 70,
 'Ö': 71,
 'Ü': 72,
 'ç': 73,
 'ö': 74,
 'ü': 75,
 'ğ': 76,
 'İ': 77,
 'ı': 78,
 'Ş': 79,
 'ş': 80,
 '‘': 81,
 '’': 82,
 '“': 83,
 '”': 84,
 '…': 85,
 '<UNK>': 86}

In [154]:
test_data == ch_tokenizer.decode(ch_tokenizer.encode(test_data))

False

In [155]:
set(test_data) - set(ch_tokenizer.stoi)

{'â'}

In [156]:
ch_tokenizer.decode(ch_tokenizer.encode(test_data)).count("<UNK>")

1